# Count Accumulation Offset Explorer: 20260629-174138

This notebook is configured for `runs/camera2/20260629-174138-sync-check`.

It loads `raw.aedat4` from the count-mode run and focuses on accumulation-window inspection only. This recording intentionally contains extra camera time before and after the DMD sequence, so the explorer aligns camera triggers to the event range and defaults to one complete DMD cycle.

The expected first cycle is 120 trigger windows: count frames `1..60` interleaved with 60 blank frames (`1, blank, 2, blank, ...`). The cycles control can still be raised to inspect repeated cycles from the longer capture.

If the run metadata contains `startup_leader.trigger_count`, those blank startup-leader triggers are skipped before event-range alignment and cycle selection. They are real TRIG_OUT_2 pulses, but they are not count/blank display frames.



In [1]:
from __future__ import annotations

from functools import lru_cache
from io import BytesIO
from math import ceil
from pathlib import Path
import json
import sys
import time

import numpy as np
try:
    from IPython.display import Image as DisplayImage, clear_output, display
except ModuleNotFoundError:
    class DisplayImage:
        def __init__(self, data=None, **kwargs):
            self.data = data

    def clear_output(*args, **kwargs):
        return None

    def display(obj):
        print(type(obj).__name__)
from PIL import Image, ImageDraw, ImageFont

try:
    import cv2
except ModuleNotFoundError:
    cv2 = None


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "dmdcontrol").is_dir() and (candidate / "runs").is_dir():
            return candidate
    raise RuntimeError("Could not find repository root from the current notebook directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RUN_DIR = REPO_ROOT / "runs" / "camera2_master_30hz_blank" / "20260728-152721-sync-check"
# C:\dev\dmdcontrol\runs\camera2_master_30hz_blank
AEDAT4_PATH = RUN_DIR / "raw.aedat4"
METADATA_PATH = RUN_DIR / "metadata.json"
SUMMARY_PATH = RUN_DIR / "summary.json"

print(f"run dir: {RUN_DIR}")
print(f"aedat4 exists: {AEDAT4_PATH.exists()}")


run dir: C:\dev\EODLA\dmdcontrol\runs\camera2_master_30hz_blank\20260728-152721-sync-check
aedat4 exists: True


In [2]:
from dmdcontrol.camera.accumulation import accumulate_events_for_triggers
from dmdcontrol.camera.timing_analysis import recommend_polarity_window, select_cycle_triggers
from dmdcontrol.camera.reprocess_aedat4 import read_aedat4_recording
from dmdcontrol.camera.runs import _events_to_arrays, _process_accumulation_triggers, _trigger_timestamps


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def _count_display_sequence(
    count_start: int,
    count_end: int,
    slots_per_frame: int,
    blank_between_frames: bool,
) -> list[str]:
    labels: list[str] = []
    counts = list(range(int(count_start), int(count_end) + 1))
    slots = max(1, int(slots_per_frame))
    for offset in range(0, len(counts), slots):
        chunk = counts[offset:offset + slots]
        for value in chunk:
            labels.append(str(value))
            if blank_between_frames:
                labels.append("blank")
    return labels


def _startup_leader_trigger_count(metadata: dict, artifact_summary: dict) -> int:
    # Paired capture now displays blank startup-leader frames after the DLPC sequencers start.
    # Their TRIG_OUT_2 pulses are real, but they are not semantic display frames, so all
    # analysis must skip them before assigning labels such as 1, blank, 2, blank, ...
    leader = metadata.get("startup_leader") or {}
    if leader.get("trigger_count") is not None:
        return int(leader.get("trigger_count") or 0)
    leader_skip = artifact_summary.get("startup_leader_skip") or {}
    return int(
        leader_skip.get("requested_trigger_count")
        if leader_skip.get("requested_trigger_count") is not None
        else leader_skip.get("skipped_trigger_count") or 0
    )


def _display_sequence_from_runtime_metadata(metadata: dict) -> list[str]:
    sequence = metadata.get("display_sequence") or {}
    labels: list[str] = []
    for frame in sequence.get("frames") or []:
        for slot in frame.get("lut_slots") or []:
            label = slot.get("semantic_label") or slot.get("semantic_role")
            if label is None:
                continue
            label = str(label)
            if label.startswith("count:"):
                label = label.split(":", 1)[1]
            labels.append(label)
    return labels


def label_for_frame(frame_index: int) -> str:
    if not display_sequence:
        return str(int(frame_index) + 1)
    return str(display_sequence[int(frame_index) % len(display_sequence)])


metadata = load_json(METADATA_PATH)
summary = load_json(SUMMARY_PATH)
artifact_summary = metadata.get("artifact_summary") or summary
# This recording is longer than one DMD sequence; default to the intended single count cycle.
DEFAULT_ACCUMULATION_CYCLES = int(globals().get("DEFAULT_ACCUMULATION_CYCLES", 1))

count_start = int(metadata.get("count_start") or 1)
count_end = int(metadata.get("count_end") or count_start)
count_slots_per_frame = int(metadata.get("count_slots_per_frame") or 1)
runtime_display_sequence = _display_sequence_from_runtime_metadata(metadata)
count_blank_between_frames = bool(metadata.get("count_blank_between_frames", False))
count_total = max(0, count_end - count_start + 1)
count_display_sequence = _count_display_sequence(
    count_start,
    count_end,
    count_slots_per_frame,
    count_blank_between_frames,
)
number_sequence = list(metadata.get("number_sequence") or [])

cycle_length = int(
    metadata.get("expected_trigger_count")
    or artifact_summary.get("trigger_cycle_limit", {}).get("cycle_length")
    or len(count_display_sequence)
    or len(number_sequence)
    or 1
)
if len(count_display_sequence) < cycle_length:
    count_display_sequence.extend(f"slot {index + 1}" for index in range(len(count_display_sequence), cycle_length))
display_sequence = (
    runtime_display_sequence[:cycle_length]
    if runtime_display_sequence
    else count_display_sequence[:cycle_length] if count_display_sequence else [str(value) for value in number_sequence]
)
blank_label_count = sum(1 for label in display_sequence if str(label).lower() == "blank")
number_label_count = sum(1 for label in display_sequence if str(label).isdigit())
expected_label_count = int(metadata.get("expected_trigger_count") or len(display_sequence) or cycle_length)
expected_number_label_count = count_total if metadata.get("test") == "a-count-b-static" else len(number_sequence)
expected_blank_label_count = count_total if count_blank_between_frames else 0
if len(display_sequence) != expected_label_count:
    raise ValueError(f"metadata/display label mismatch: {len(display_sequence)} labels for {expected_label_count} expected triggers")
if number_label_count != expected_number_label_count or blank_label_count != expected_blank_label_count:
    raise ValueError(
        "metadata count/blank mismatch: "
        f"got {number_label_count} count labels + {blank_label_count} blank labels; "
        f"expected {expected_number_label_count} + {expected_blank_label_count}"
    )
grid_cols = int(globals().get("GRID_COLS", min(5, max(1, cycle_length))))

metadata_cycles = metadata.get("accumulation_cycles")
default_cycles = int(metadata_cycles if metadata_cycles is not None else DEFAULT_ACCUMULATION_CYCLES)
metadata_default_window_us = int(
    metadata.get("accumulation_window_us")
    or metadata.get("exposure_us")
    or artifact_summary.get("window_us")
    or 8000
)
default_window_us = metadata_default_window_us
metadata_offset_us = metadata.get("accumulation_start_offset_us")
summary_offset_us = artifact_summary.get("window_start_offset_us")
default_dark_time_us = int(metadata.get("dark_time_us") or 0)
default_offset_us = int(
    metadata_offset_us
    if metadata_offset_us is not None
    else summary_offset_us if summary_offset_us is not None else 0
)
timing_a = metadata.get("timing_a") or {}
trigger_period_us = int(
    round(timing_a.get("frame_period_us") or default_window_us + default_dark_time_us or 16667)
)
startup_leader_trigger_count = _startup_leader_trigger_count(metadata, artifact_summary)


t0 = time.perf_counter()
recording = read_aedat4_recording(AEDAT4_PATH)
event_arrays = _events_to_arrays(recording.events)
event_timestamps = event_arrays["t"]
event_polarities = event_arrays["p"]
FORCE_AEDAT4_TRIGGERS = True
trigger_source = "raw AEDAT4 rising triggers from DMD A"
default_trigger_stages = _process_accumulation_triggers(
    recording.triggers,
    event_timestamps,
    window_us=metadata_default_window_us,
    window_start_offset_us=default_offset_us,
    max_accumulation_triggers=None,
    trigger_cycle_length=cycle_length,
    accumulation_cycles=None,
    startup_leader_trigger_count=startup_leader_trigger_count,
)
semantic_trigger_timestamps = _trigger_timestamps(default_trigger_stages.final)
available_cycles = int(
    default_trigger_stages.cycle_limit_metadata.get("available_full_cycles") or 0
)
timing_recommendation = None
timing_recommendation_error = None
default_cycle_start = 0
analysis_window_us = min(4000, trigger_period_us)
if available_cycles:
    try:
        timing_recommendation = recommend_polarity_window(
            event_timestamps,
            event_polarities,
            semantic_trigger_timestamps,
            display_sequence,
            trigger_period_us=trigger_period_us,
            window_us=analysis_window_us,
            offset_step_us=250,
            skip_first_cycle_when_possible=True,
        )
    except ValueError as exc:
        timing_recommendation_error = str(exc)
if timing_recommendation is not None:
    default_cycle_start = timing_recommendation.selected.cycle_index
    default_offset_us = timing_recommendation.selected.offset_us
    default_window_us = timing_recommendation.selected.window_us
default_cycles = max(
    1,
    min(default_cycles, max(1, available_cycles - default_cycle_start)),
)
offset_min_us = 0
offset_max_us = max(0, trigger_period_us - default_window_us)
load_s = time.perf_counter() - t0


width, height = recording.resolution
print(f"run dir: {RUN_DIR}")
print(f"loaded AEDAT4 in {load_s:.2f}s")
print(f"resolution: {width} x {height}")
print(f"events: {recording.stats['event_count']:,}; triggers: {recording.stats['trigger_count']:,}")
print(f"trigger edges: {recording.stats['trigger_edges']}")
print(f"trigger source for accumulation: {trigger_source}; available full cycles after leader/alignment: {available_cycles}")
print(f"startup leader trigger skip: {startup_leader_trigger_count}")
print(f"count range: {count_start}..{count_end}; slots/frame: {count_slots_per_frame}; blank-between: {count_blank_between_frames}")
print(f"cycle length: {cycle_length}; grid columns: {grid_cols}; default cycles: {default_cycles}; default window: {default_window_us} us")
print(f"expected one-cycle labels: {number_label_count} count frames + {blank_label_count} blank frames")
print(f"metadata validation: {expected_label_count} expected triggers from {metadata.get('test')}; labels match metadata")
if timing_recommendation is not None:
    selected_score = timing_recommendation.selected
    print(
        f"recommended stable cycle: {selected_score.cycle_index + 1}; "
        f"window: +{selected_score.offset_us}..+{selected_score.offset_us + selected_score.window_us} us; "
        f"polarity accuracy: {selected_score.polarity_accuracy:.3f}"
    )
    print(
        "per-cycle best: "
        + ", ".join(
            f"cycle {score.cycle_index + 1}=+{score.offset_us} us/{score.polarity_accuracy:.3f}"
            for score in timing_recommendation.per_cycle_best
        )
    )
else:
    print(f"timing recommendation unavailable: {timing_recommendation_error}")
print(
    f"default offset: {default_offset_us} us; default window: {default_window_us} us; "
    f"offset slider range: {offset_min_us}..{offset_max_us} us"
)
print(f"metadata offset: {metadata_offset_us}; summary offset: {summary_offset_us}")
print(f"first 16 labels: {display_sequence[:16]}")


run dir: C:\dev\EODLA\dmdcontrol\runs\camera2_master_30hz_blank\20260728-152721-sync-check
loaded AEDAT4 in 15.24s
resolution: 346 x 260
events: 2,418,139; triggers: 1,362
trigger edges: {'rising': 599, 'triggertype.external_signal_pulse': 599, 'triggertype.aps_exposure_end': 41, 'triggertype.aps_frame_end': 41, 'triggertype.aps_frame_start': 41, 'triggertype.aps_exposure_start': 41}
trigger source for accumulation: raw AEDAT4 rising triggers from DMD A; available full cycles after leader/alignment: 26
startup leader trigger skip: 16
count range: 5..15; slots/frame: 1; blank-between: True
cycle length: 22; grid columns: 5; default cycles: 18; default window: 4000 us
expected one-cycle labels: 11 count frames + 11 blank frames
metadata validation: 22 expected triggers from a-count-b-static; labels match metadata
recommended stable cycle: 9; window: +11000..+15000 us; polarity accuracy: 0.631
per-cycle best: cycle 1=+9000 us/0.589, cycle 2=+11250 us/0.603, cycle 3=+7750 us/0.593, cycle 4

In [3]:
@lru_cache(maxsize=16)
def accumulate_cached(
    offset_us: int,
    cycle_start: int,
    cycles: int,
    window_us: int,
    polarity_mode: str,
):
    requested_cycles = int(cycles)
    requested_cycle_start = int(cycle_start)
    stages = _process_accumulation_triggers(
        recording.triggers,
        event_timestamps,
        window_us=int(window_us),
        window_start_offset_us=int(offset_us),
        max_accumulation_triggers=None,
        trigger_cycle_length=cycle_length,
        accumulation_cycles=None,
        startup_leader_trigger_count=startup_leader_trigger_count,
    )
    all_selected = stages.final
    all_trigger_timestamps = _trigger_timestamps(all_selected)
    selected_trigger_timestamps = select_cycle_triggers(
        all_trigger_timestamps,
        cycle_length=cycle_length,
        cycle_index=requested_cycle_start,
        cycles=requested_cycles,
    )
    start = requested_cycle_start * cycle_length
    stop = start + requested_cycles * cycle_length
    selected = all_selected[start:stop]
    alignment = stages.alignment_metadata
    cycle_info = {
        **stages.cycle_limit_metadata,
        "selected_cycle_start": requested_cycle_start,
        "selected_cycles": requested_cycles,
        "selected_trigger_count": len(selected),
    }
    frames = accumulate_events_for_triggers(
        recording.events,
        selected,
        resolution=recording.resolution,
        window_us=int(window_us),
        polarity_mode=str(polarity_mode),
        window_start_offset_us=int(offset_us),
    )
    return frames, selected_trigger_timestamps, alignment, cycle_info


def frame_values(frames: np.ndarray, view: str, tone: str, gamma: float) -> tuple[np.ndarray, bool]:
    source = frames.astype(np.float32, copy=False)
    if view == "signed":
        return source, True
    if view == "positive":
        values = np.maximum(source, 0)
    elif view == "negative":
        values = np.maximum(-source, 0)
    else:
        values = np.abs(source)
    if tone == "log":
        values = np.log1p(values)
    elif tone == "gamma":
        values = np.power(values, float(gamma))
    return values, False


def auto_roi(frames: np.ndarray, pad: int, flip_x: bool, crop_mode: str) -> tuple[slice, slice]:
    if crop_mode == "full":
        return slice(0, frames.shape[1]), slice(0, frames.shape[2])
    source = frames[:, :, ::-1] if flip_x else frames
    mask = np.any(np.abs(source) > 0, axis=0)
    if not np.any(mask):
        return slice(0, frames.shape[1]), slice(0, frames.shape[2])
    ys, xs = np.where(mask)
    y0 = max(0, int(ys.min()) - int(pad))
    y1 = min(frames.shape[1], int(ys.max()) + int(pad) + 1)
    x0 = max(0, int(xs.min()) - int(pad))
    x1 = min(frames.shape[2], int(xs.max()) + int(pad) + 1)
    if crop_mode == "square":
        side = max(y1 - y0, x1 - x0)
        cy = (y0 + y1) // 2
        cx = (x0 + x1) // 2
        y0 = max(0, min(frames.shape[1] - side, cy - side // 2))
        x0 = max(0, min(frames.shape[2] - side, cx - side // 2))
        y1 = min(frames.shape[1], y0 + side)
        x1 = min(frames.shape[2], x0 + side)
    return slice(y0, y1), slice(x0, x1)


def dilate_plane(plane: np.ndarray, dot_size: int) -> np.ndarray:
    dot_size = int(dot_size)
    if dot_size <= 1:
        return plane
    if cv2 is not None:
        kernel = np.ones((dot_size, dot_size), dtype=np.uint8)
        return cv2.dilate(plane.astype(np.float32, copy=False), kernel)
    padded = np.pad(plane, dot_size // 2, mode="edge")
    out = np.zeros_like(plane)
    for dy in range(dot_size):
        for dx in range(dot_size):
            out = np.maximum(out, padded[dy:dy + plane.shape[0], dx:dx + plane.shape[1]])
    return out


def scale_limit(values: np.ndarray, signed: bool, percentile: float) -> float:
    source = np.abs(values) if signed else values
    nonzero = source[source > 0]
    if nonzero.size == 0:
        return 1.0
    return max(float(np.percentile(nonzero, float(percentile))), 1e-6)


def frame_rgb(values: np.ndarray, signed: bool, limit: float, dot_size: int) -> np.ndarray:
    if signed:
        pos = dilate_plane(np.maximum(values, 0), dot_size)
        neg = dilate_plane(np.maximum(-values, 0), dot_size)
        pos8 = np.clip(pos / limit * 255, 0, 255).astype(np.uint8)
        neg8 = np.clip(neg / limit * 255, 0, 255).astype(np.uint8)
        rgb = np.zeros((*values.shape, 3), dtype=np.uint8)
        rgb[..., 0] = pos8
        rgb[..., 1] = neg8
        rgb[..., 2] = neg8
        return rgb
    plane = dilate_plane(values, dot_size)
    gray = np.clip(plane / limit * 255, 0, 255).astype(np.uint8)
    return np.repeat(gray[..., None], 3, axis=2)


def resize_nearest(rgb: np.ndarray, scale: int) -> Image.Image:
    image = Image.fromarray(rgb, mode="RGB")
    if int(scale) <= 1:
        return image
    return image.resize((image.width * int(scale), image.height * int(scale)), Image.Resampling.NEAREST)


def png_bytes(image: Image.Image) -> bytes:
    buffer = BytesIO()
    image.save(buffer, format="PNG", optimize=False)
    return buffer.getvalue()


In [4]:
FONT = ImageFont.load_default()


def render_fast_sheet(
    offset_us: int = default_offset_us,
    cycle_start: int = default_cycle_start,
    cycles: int = default_cycles,
    window_us: int = default_window_us,
    polarity_mode: str = "signed",
    view: str = "positive",
    tone: str = "log",
    gamma: float = 0.65,
    contrast: str = "per-frame",
    vmax_percentile: float = 99.0,
    crop_mode: str = "square",
    crop_pad: int = 28,
    dot_size: int = 2,
    tile_scale: int = 2,
    focus_frame: int = 1,
    focus_scale: int = 4,
    flip_x: bool = True,
):
    start = time.perf_counter()
    frames, trigger_ts, alignment, cycle_info = accumulate_cached(
        int(offset_us), int(cycle_start), int(cycles), int(window_us), str(polarity_mode)
    )
    frames_for_view = frames[:, :, ::-1] if flip_x else frames
    y_slice, x_slice = auto_roi(frames, crop_pad, flip_x=flip_x, crop_mode=crop_mode)
    cropped = frames_for_view[:, y_slice, x_slice]
    values, signed = frame_values(cropped, view=view, tone=tone, gamma=gamma)
    frame_count = values.shape[0]
    cols = max(1, int(grid_cols))
    rows = max(1, ceil(frame_count / cols))

    if contrast == "global":
        global_limit = scale_limit(values, signed=signed, percentile=vmax_percentile)
    else:
        global_limit = None

    tile_images = []
    for index in range(frame_count):
        limit = global_limit or scale_limit(values[index], signed=signed, percentile=vmax_percentile)
        tile_images.append(resize_nearest(frame_rgb(values[index], signed, limit, dot_size), tile_scale))

    tile_w = tile_images[0].width if tile_images else 1
    tile_h = tile_images[0].height if tile_images else 1
    label_h = 26
    header_h = 58
    gap = 8
    focus_index = max(0, min(frame_count - 1, int(focus_frame) - 1)) if frame_count else 0
    focus_limit = global_limit or scale_limit(values[focus_index], signed=signed, percentile=vmax_percentile)
    focus_image = resize_nearest(frame_rgb(values[focus_index], signed, focus_limit, max(1, int(dot_size))), focus_scale)

    sheet_w = cols * tile_w + (cols - 1) * gap
    sheet_h = rows * (tile_h + label_h) + (rows - 1) * gap
    canvas_w = max(sheet_w, focus_image.width) + 20
    canvas_h = header_h + focus_image.height + 22 + sheet_h + 24
    canvas = Image.new("RGB", (canvas_w, canvas_h), (14, 14, 14))
    draw = ImageDraw.Draw(canvas)

    counts = np.count_nonzero(np.abs(frames) > 0, axis=(1, 2)) if frame_count else np.array([], dtype=int)
    roi_text = f"roi y={y_slice.start}:{y_slice.stop} x={x_slice.start}:{x_slice.stop}"
    title = (
        f"offset {int(offset_us)} us | window {int(window_us)} us | {frame_count} frames | "
        f"{polarity_mode}/{view}/{tone} | contrast={contrast} | dot={int(dot_size)}"
    )
    draw.text((10, 8), title, fill=(235, 235, 235), font=FONT)
    draw.text((10, 28), roi_text, fill=(180, 180, 180), font=FONT)
    if counts.size:
        draw.text(
            (10, 44),
            f"nonzero pixels: first={int(counts[0])}, min={int(counts.min())}, median={int(np.median(counts))}, max={int(counts.max())}",
            fill=(180, 180, 180),
            font=FONT,
        )

    focus_x = (canvas_w - focus_image.width) // 2
    focus_y = header_h
    canvas.paste(focus_image, (focus_x, focus_y))
    focus_cycle = int(cycle_start) + focus_index // max(1, int(cycle_length)) + 1
    focus_label = label_for_frame(focus_index)
    draw.text(
        (10, focus_y + focus_image.height + 4),
        f"focus frame {focus_index + 1}: expected {focus_label}, cycle {focus_cycle}, nonzero={int(counts[focus_index]) if counts.size else 0}",
        fill=(235, 235, 235),
        font=FONT,
    )

    first_trigger = int(trigger_ts[0]) if len(trigger_ts) else 0
    grid_y = focus_y + focus_image.height + 26
    for index, image in enumerate(tile_images):
        row = index // cols
        col = index % cols
        x = 10 + col * (tile_w + gap)
        y = grid_y + row * (tile_h + label_h + gap)
        canvas.paste(image, (x, y + label_h))
        index % cols
        cycle = index // max(1, int(cycle_length)) + 1
        label = label_for_frame(index)
        dt = int(trigger_ts[index] - first_trigger) if len(trigger_ts) else 0
        color = (255, 245, 170) if index == focus_index else (220, 220, 220)
        draw.text((x, y), f"{index + 1}: {label} c{cycle} +{dt}us", fill=color, font=FONT)
        if counts.size:
            draw.text((x, y + 12), f"nz={int(counts[index])}", fill=(170, 170, 170), font=FONT)

    elapsed_ms = (time.perf_counter() - start) * 1000
    draw.text((canvas_w - 115, 8), f"{elapsed_ms:.0f} ms", fill=(120, 210, 120), font=FONT)
    return canvas, frames, counts, alignment, cycle_info


image, frames, counts, alignment, cycle_info = render_fast_sheet(offset_us=default_offset_us)
display(DisplayImage(data=png_bytes(image)))
print(
    "default trigger alignment: "
    f"selected={cycle_info.get('selected_trigger_count')} / cycle_length={cycle_info.get('cycle_length')}; "
    f"dropped_before={alignment.get('dropped_before_event_count')}, "
    f"dropped_after={alignment.get('dropped_after_event_count')}"
)


default trigger alignment: selected=396 / cycle_length=22; dropped_before=0, dropped_after=0


## Interactive Controls

Defaults are tuned to make sparse event frames visible and to show one aligned 120-trigger count cycle first: ROI crop, per-frame contrast, log tone, and dot dilation. Switch contrast to `global` when you want brightness comparisons to be physically meaningful across frames.


In [ ]:
try:
    import ipywidgets as widgets
    HAVE_WIDGETS = True
except ModuleNotFoundError:
    HAVE_WIDGETS = False
    print("ipywidgets is not installed in this kernel. Run `%pip install ipywidgets`, restart the kernel, then rerun this notebook.")


ENABLE_INTERACTIVE_WIDGETS = bool(globals().get("ENABLE_INTERACTIVE_WIDGETS", True))

if HAVE_WIDGETS and ENABLE_INTERACTIVE_WIDGETS:
    offset_slider = widgets.IntSlider(
        value=default_offset_us,
        min=offset_min_us,
        max=offset_max_us,
        step=1,
        description="offset us",
        continuous_update=True,
        layout=widgets.Layout(width="560px"),
    )
    cycle_start_slider = widgets.IntSlider(
        value=default_cycle_start + 1,
        min=1,
        max=max(1, available_cycles),
        step=1,
        description="start cycle",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    cycles_slider = widgets.IntSlider(
        value=default_cycles,
        min=1,
        max=max(1, available_cycles - default_cycle_start),
        step=1,
        description="cycles",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    window_slider = widgets.IntSlider(
        value=default_window_us,
        min=100,
        max=max(2500, default_window_us * 2),
        step=25,
        description="window us",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    focus_slider = widgets.IntSlider(
        value=1,
        min=1,
        max=max(1, default_cycles * cycle_length),
        step=1,
        description="focus",
        continuous_update=True,
        layout=widgets.Layout(width="560px"),
    )
    polarity_dropdown = widgets.Dropdown(value="signed", options=["ignore", "signed", "positive"], description="accum")
    view_dropdown = widgets.Dropdown(value="positive", options=["magnitude", "positive", "negative", "signed"], description="view")
    tone_dropdown = widgets.Dropdown(value="log", options=["log", "gamma", "linear"], description="tone")
    contrast_dropdown = widgets.Dropdown(value="per-frame", options=["per-frame", "global"], description="contrast")
    crop_dropdown = widgets.Dropdown(value="square", options=["square", "auto", "full"], description="crop")
    gamma_slider = widgets.FloatSlider(value=0.65, min=0.2, max=1.5, step=0.05, description="gamma", continuous_update=False, layout=widgets.Layout(width="560px"))
    vmax_slider = widgets.FloatSlider(value=99.0, min=90.0, max=100.0, step=0.1, description="vmax %", continuous_update=False, layout=widgets.Layout(width="560px"))
    pad_slider = widgets.IntSlider(value=28, min=0, max=80, step=2, description="crop pad", continuous_update=False, layout=widgets.Layout(width="560px"))
    dot_slider = widgets.IntSlider(value=2, min=1, max=6, step=1, description="dot size", continuous_update=True, layout=widgets.Layout(width="560px"))
    tile_scale_slider = widgets.IntSlider(value=2, min=1, max=4, step=1, description="tile scale", continuous_update=False, layout=widgets.Layout(width="560px"))
    focus_scale_slider = widgets.IntSlider(value=4, min=2, max=8, step=1, description="focus scale", continuous_update=False, layout=widgets.Layout(width="560px"))
    flip_checkbox = widgets.Checkbox(value=True, description="flip x for viewing")
    out = widgets.Output()

    def sync_focus_max(*_):
        cycles_slider.max = max(1, available_cycles - (cycle_start_slider.value - 1))
        if cycles_slider.value > cycles_slider.max:
            cycles_slider.value = cycles_slider.max
        focus_slider.max = max(1, cycles_slider.value * cycle_length)
        if focus_slider.value > focus_slider.max:
            focus_slider.value = focus_slider.max

    def redraw(_=None):
        sync_focus_max()
        with out:
            clear_output(wait=True)
            image, frames, counts, alignment, cycle_info = render_fast_sheet(
                offset_us=offset_slider.value,
                cycle_start=cycle_start_slider.value - 1,
                cycles=cycles_slider.value,
                window_us=window_slider.value,
                polarity_mode=polarity_dropdown.value,
                view=view_dropdown.value,
                tone=tone_dropdown.value,
                gamma=gamma_slider.value,
                contrast=contrast_dropdown.value,
                vmax_percentile=vmax_slider.value,
                crop_mode=crop_dropdown.value,
                crop_pad=pad_slider.value,
                dot_size=dot_slider.value,
                tile_scale=tile_scale_slider.value,
                focus_frame=focus_slider.value,
                focus_scale=focus_scale_slider.value,
                flip_x=flip_checkbox.value,
            )
            display(DisplayImage(data=png_bytes(image)))

    for widget in [
        offset_slider,
        cycle_start_slider,
        cycles_slider,
        window_slider,
        focus_slider,
        polarity_dropdown,
        view_dropdown,
        tone_dropdown,
        contrast_dropdown,
        crop_dropdown,
        gamma_slider,
        vmax_slider,
        pad_slider,
        dot_slider,
        tile_scale_slider,
        focus_scale_slider,
        flip_checkbox,
    ]:
        widget.observe(redraw, names="value")

    controls = widgets.VBox([
        offset_slider,
        cycle_start_slider,
        cycles_slider,
        window_slider,
        focus_slider,
        widgets.HBox([polarity_dropdown, view_dropdown, tone_dropdown, contrast_dropdown, crop_dropdown]),
        gamma_slider,
        vmax_slider,
        pad_slider,
        widgets.HBox([dot_slider, tile_scale_slider, focus_scale_slider, flip_checkbox]),
    ])
    display(controls)
    display(out)
    redraw()
else:
    image, frames, counts, alignment, cycle_info = render_fast_sheet(offset_us=default_offset_us)
    display(DisplayImage(data=png_bytes(image)))


Output()

## Offset Score Table

This quick numerical scan keeps the count/blank trigger order unchanged. It reports nonzero-pixel counts for candidate accumulation offsets using the current window length and the default one-cycle trigger selection.


In [6]:
def offset_score_table(
    offsets=range(0, max(0, trigger_period_us - default_window_us) + 1, 250),
    cycle_start=default_cycle_start,
    cycles=default_cycles,
    window_us=default_window_us,
    polarity_mode="signed",
):
    rows = []
    for offset in offsets:
        frames, trigger_ts, alignment, cycle_info = accumulate_cached(
            int(offset), int(cycle_start), int(cycles), int(window_us), str(polarity_mode)
        )
        counts = np.count_nonzero(np.abs(frames) > 0, axis=(1, 2))
        first_cycle = counts[:cycle_length] if counts.size else np.array([], dtype=int)
        rows.append({
            "offset_us": int(offset),
            "frame1": int(counts[0]) if counts.size else 0,
            "first_cycle_min": int(first_cycle.min()) if first_cycle.size else 0,
            "first_cycle_median": int(np.median(first_cycle)) if first_cycle.size else 0,
            "all_min": int(counts.min()) if counts.size else 0,
            "all_median": int(np.median(counts)) if counts.size else 0,
            "all_max": int(counts.max()) if counts.size else 0,
        })
    return sorted(rows, key=lambda row: (row["first_cycle_min"], row["frame1"], row["all_median"]), reverse=True)


scores = offset_score_table()
for row in scores[:12]:
    print(row)


{'offset_us': 12250, 'frame1': 1052, 'first_cycle_min': 838, 'first_cycle_median': 922, 'all_min': 787, 'all_median': 955, 'all_max': 1358}
{'offset_us': 12000, 'frame1': 1049, 'first_cycle_min': 834, 'first_cycle_median': 929, 'all_min': 778, 'all_median': 958, 'all_max': 1331}
{'offset_us': 12500, 'frame1': 1039, 'first_cycle_min': 830, 'first_cycle_median': 930, 'all_min': 789, 'all_median': 960, 'all_max': 1353}
{'offset_us': 11750, 'frame1': 1037, 'first_cycle_min': 815, 'first_cycle_median': 934, 'all_min': 786, 'all_median': 958, 'all_max': 1339}
{'offset_us': 10000, 'frame1': 1085, 'first_cycle_min': 812, 'first_cycle_median': 910, 'all_min': 751, 'all_median': 943, 'all_max': 1373}
{'offset_us': 10250, 'frame1': 1085, 'first_cycle_min': 810, 'first_cycle_median': 915, 'all_min': 741, 'all_median': 943, 'all_max': 1346}
{'offset_us': 10500, 'frame1': 1090, 'first_cycle_min': 808, 'first_cycle_median': 914, 'all_min': 766, 'all_median': 946, 'all_max': 1331}
{'offset_us': 11000,

## Trim Displayed Trigger Window AEDAT4

Write a compact AEDAT4 file containing only the events around the displayed trigger frames. The default output keeps one full displayed count cycle, 60 number frames plus 60 blank frames for this run, and includes a 2000 us timestamp buffer before the first trigger and after the last trigger.


In [7]:
TRIMMED_AEDAT4_TRIGGER_COUNT = max(1, int(default_cycles) * int(cycle_length))
TRIMMED_AEDAT4_BUFFER_US = 2000
TRIMMED_AEDAT4_PATH = RUN_DIR / f"trimmed_displayed_{TRIMMED_AEDAT4_TRIGGER_COUNT}_triggers_plus_{TRIMMED_AEDAT4_BUFFER_US}us.aedat4"
TRIMMED_AEDAT4_METADATA_PATH = RUN_DIR / f"trimmed_displayed_{TRIMMED_AEDAT4_TRIGGER_COUNT}_triggers_plus_{TRIMMED_AEDAT4_BUFFER_US}us.json"


def displayed_trigger_trim_window(trigger_ts, frame_count: int = 30, buffer_us: int = 2000):
    trigger_ts = np.asarray(trigger_ts, dtype=np.int64)
    if trigger_ts.size == 0:
        raise ValueError("at least one trigger timestamp is required")
    selected_count = max(1, min(int(frame_count), int(trigger_ts.size)))
    selected = trigger_ts[:selected_count]
    buffer_us = max(0, int(buffer_us))
    first_trigger_us = int(selected[0])
    last_trigger_us = int(selected[-1])
    return {
        "start_us": int(first_trigger_us - buffer_us),
        "stop_us": int(last_trigger_us + buffer_us),
        "first_trigger_us": first_trigger_us,
        "last_trigger_us": last_trigger_us,
        "selected_trigger_count": int(selected_count),
        "buffer_us": int(buffer_us),
    }


def trim_event_arrays_for_time_window(arrays, window):
    timestamps = np.asarray(arrays["t"], dtype=np.int64)
    keep = (timestamps >= int(window["start_us"])) & (timestamps < int(window["stop_us"]))
    return {
        "x": np.asarray(arrays["x"])[keep].astype(np.int64, copy=False),
        "y": np.asarray(arrays["y"])[keep].astype(np.int64, copy=False),
        "t": timestamps[keep],
        "p": np.asarray(arrays["p"])[keep].astype(np.bool_, copy=False),
    }


def trigger_type_for_edge(edge: str):
    import dv_processing as dv

    edge_text = str(edge).lower()
    if "falling" in edge_text:
        return dv.TriggerType.EXTERNAL_SIGNAL_FALLING_EDGE
    if "pulse" in edge_text:
        return dv.TriggerType.EXTERNAL_SIGNAL_PULSE
    return dv.TriggerType.EXTERNAL_SIGNAL_RISING_EDGE


def trigger_timestamp(record) -> int:
    value = getattr(record, "timestamp", None)
    if value is None and isinstance(record, dict):
        value = record.get("timestamp")
    if callable(value):
        value = value()
    return int(value)


def trigger_edge(record) -> str:
    value = getattr(record, "edge", None)
    if value is None and isinstance(record, dict):
        value = record.get("edge")
    if callable(value):
        value = value()
    return "rising" if value is None else str(value)


def write_trimmed_display_aedat4(
    output_path,
    metadata_path,
    arrays,
    selected_triggers,
    resolution,
    source_aedat4_path,
    buffer_us: int = 2000,
):
    import dv_processing as dv

    selected_triggers = list(selected_triggers)
    if not selected_triggers:
        raise ValueError("selected_triggers must not be empty")
    trigger_ts = np.asarray([trigger_timestamp(trigger) for trigger in selected_triggers], dtype=np.int64)
    window = displayed_trigger_trim_window(trigger_ts, frame_count=len(selected_triggers), buffer_us=int(buffer_us))
    trimmed_arrays = trim_event_arrays_for_time_window(arrays, window)

    output_path = Path(output_path)
    metadata_path = Path(metadata_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        output_path.unlink()

    width, height = (int(resolution[0]), int(resolution[1]))
    config = dv.io.MonoCameraWriter.Config("trimmed_displayed_trigger_window")
    config.addEventStream((width, height), streamName="events", source="trimmed-events")
    config.addTriggerStream(streamName="triggers", source="trimmed-triggers")
    writer = dv.io.MonoCameraWriter(str(output_path), config)

    event_store = dv.EventStore()
    for timestamp, x, y, polarity in zip(
        trimmed_arrays["t"],
        trimmed_arrays["x"],
        trimmed_arrays["y"],
        trimmed_arrays["p"],
        strict=False,
    ):
        event_store.push_back(int(timestamp), int(x), int(y), bool(polarity))
    writer.writeEvents(event_store, streamName="events")

    trigger_packet = dv.TriggerPacket()
    for trigger in selected_triggers:
        trigger_packet.elements.append(dv.Trigger(trigger_timestamp(trigger), trigger_type_for_edge(trigger_edge(trigger))))
    writer.writeTriggerPacket(trigger_packet, streamName="triggers")
    del writer

    metadata = {
        "source_aedat4": str(source_aedat4_path),
        "output_aedat4": str(output_path),
        "selected_trigger_count": int(len(selected_triggers)),
        "buffer_us": int(buffer_us),
        "start_us": int(window["start_us"]),
        "stop_us": int(window["stop_us"]),
        "first_trigger_us": int(window["first_trigger_us"]),
        "last_trigger_us": int(window["last_trigger_us"]),
        "event_count": int(len(trimmed_arrays["t"])),
        "resolution": [width, height],
    }
    metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8")
    return metadata


trim_cycles = max(1, ceil(int(TRIMMED_AEDAT4_TRIGGER_COUNT) / max(1, int(cycle_length))))
trim_stages = _process_accumulation_triggers(
    recording.triggers,
    event_arrays["t"],
    window_us=default_window_us,
    window_start_offset_us=default_offset_us,
    max_accumulation_triggers=int(TRIMMED_AEDAT4_TRIGGER_COUNT),
    trigger_cycle_length=cycle_length,
    accumulation_cycles=trim_cycles,
    startup_leader_trigger_count=startup_leader_trigger_count,
)
trim_selected_triggers = list(trim_stages.final[:int(TRIMMED_AEDAT4_TRIGGER_COUNT)])
trim_metadata = write_trimmed_display_aedat4(
    TRIMMED_AEDAT4_PATH,
    TRIMMED_AEDAT4_METADATA_PATH,
    event_arrays,
    trim_selected_triggers,
    (width, height),
    AEDAT4_PATH,
    buffer_us=TRIMMED_AEDAT4_BUFFER_US,
)
print(json.dumps(trim_metadata, indent=2, sort_keys=True))


{
  "buffer_us": 2000,
  "event_count": 1594499,
  "first_trigger_us": 1785266846725065,
  "last_trigger_us": 1785266853308351,
  "output_aedat4": "C:\\dev\\EODLA\\dmdcontrol\\runs\\camera2_master_30hz_blank\\20260728-152721-sync-check\\trimmed_displayed_396_triggers_plus_2000us.aedat4",
  "resolution": [
    346,
    260
  ],
  "selected_trigger_count": 396,
  "source_aedat4": "C:\\dev\\EODLA\\dmdcontrol\\runs\\camera2_master_30hz_blank\\20260728-152721-sync-check\\raw.aedat4",
  "start_us": 1785266846723065,
  "stop_us": 1785266853310351
}
